In [1]:
#câu 1:
def bayes():
    # Xác suất thuộc từng nhóm
    p_deu = 0.50
    p_don = 0.30
    p_it = 0.20

    # Xác suất qua môn của từng nhóm
    p_qua_deu = 0.95
    p_qua_don = 0.70
    p_qua_it = 0.30

    # Xác suất qua môn chung
    p_qua = (
        p_deu * p_qua_deu
        + p_don * p_qua_don
        + p_it * p_qua_it
    )

    # Áp dụng định lý Bayes
    kq_deu = p_deu * p_qua_deu / p_qua
    kq_don = p_don * p_qua_don / p_qua
    kq_it = p_it * p_qua_it / p_qua

    print(f"Nhóm học đều: {kq_deu:.4f}")
    print(f"Nhóm học dồn: {kq_don:.4f}")
    print(f"Nhóm ít học:  {kq_it:.4f}")

    # Kiểm tra tổng xác suất
    tong = kq_deu + kq_don + kq_it
    print(f"Tổng xác suất: {tong:.4f}")


bayes()

Nhóm học đều: 0.6376
Nhóm học dồn: 0.2819
Nhóm ít học:  0.0805
Tổng xác suất: 1.0000


nhận xét: Khi biết một sinh viên đã qua môn, xác suất bạn đó thuộc nhóm học đều là khoảng 63.76%, nhóm học dồn là 28.19%, và nhóm ít học là 8.05%. Tổng ba xác suất bằng 1, nên kết quả được kiểm tra đúng.

In [12]:
#CÂU 2:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Doc du lieu
df = pd.read_csv("/Users/buixuankhanh/Documents/machine-learning/03_Navie_Bayes/data/tin_nhan.csv")

# Bien doi tin nhan thanh bang dem tus
vec = CountVectorizer(token_pattern=r"(?u)\b\S+")
X = vec.fit_transform(df["noi_dung"])

# Huấn luyện trên toàn bộ 300 tin nhắn
y = df["la_rac"]
mo_hinh = MultinomialNB()
mo_hinh.fit(X, y)

# Tính hiệu log xác suất giữa lớp 1 và lớp 0
hieu = mo_hinh.feature_log_prob_[1] - mo_hinh.feature_log_prob_[0]

# Lấy danh sách từ trong từ điển
tu = vec.get_feature_names_out()

# 5 từ tố cáo tin rác mạnh nhất
print("Nam tu to cao tin rac manh nhat:")
for i in np.argsort(hieu)[-5:][::-1]:
    print(f" {tu[i]}: {hieu[i]:.4f}")

print()

# 5 từ cho thấy tin thường mạnh nhất
print("Nam tu cho thay tin thuong manh nhat:")
for i in np.argsort(hieu)[:5]:
    print(f" {tu[i]}: {hieu[i]:.4f}")

Nam tu to cao tin rac manh nhat:
 mãi: 3.2034
 tay: 2.9056
 ngay: 2.6618
 miễn: 2.4795
 trúng: 2.3742

Nam tu cho thay tin thuong manh nhat:
 lớp: -2.6122
 em: -2.5618
 bài: -2.2946
 báo: -2.2741
 phòng: -2.2741


Nhận xét kết quả Naive Bayes
- 5 từ tố cáo tin rác mạnh nhất: “mãi”, “tay”, “ngay”, “miễn”, “trúng” có hiệu log xác suất dương. Những từ này xuất hiện có xu hướng nhiều hơn trong tin nhắn rác, đặc biệt là nội dung quảng cáo, khuyến mãi hoặc thông báo trúng thưởng.
- 5 từ cho thấy tin thường mạnh nhất: “lớp”, “em”, “bài”, “báo”, “phòng” có hiệu log xác suất âm. Những từ này có xu hướng xuất hiện nhiều hơn trong tin nhắn thông thường, chẳng hạn như trao đổi về học tập hoặc công việc.
Kết luận: Mô hình Naive Bayes sử dụng tần suất xuất hiện của từ để phân biệt tin nhắn rác và tin nhắn thông thường. Từ có hiệu log xác suất dương nghiêng về tin rác, còn từ có hiệu âm nghiêng về tin thường. Tuy nhiên, đây chỉ là dấu hiệu thống kê, không có nghĩa một từ luôn xác định được loại tin nhắn.

In [17]:

# câu 3:

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Đọc kho tin nhắn
df = pd.read_csv(
    "/Users/buixuankhanh/Documents/machine-learning/03_Navie_Bayes/data/tin_nhan.csv"
)

# Chia dữ liệu thành tập huấn luyện và tập kiểm tra
van_ban_huan_luyen, van_ban_kiem_tra, nhan_huan_luyen, nhan_kiem_tra = train_test_split(
    df["noi_dung"],
    df["la_rac"],
    test_size=0.25,
    random_state=17,
    stratify=df["la_rac"]
)

# Biến đổi tin nhắn thành bảng đếm từ
bo_dem_tu = CountVectorizer(token_pattern=r"(?u)\b\S+")
X_huan_luyen = bo_dem_tu.fit_transform(van_ban_huan_luyen)
X_kiem_tra = bo_dem_tu.transform(van_ban_kiem_tra)

# Thử các giá trị alpha
for he_so_alpha in [0.001, 0.1, 1.0, 10.0, 100.0]:
    mo_hinh = MultinomialNB(alpha=he_so_alpha)
    mo_hinh.fit(X_huan_luyen, nhan_huan_luyen)

    nhan_du_doan = mo_hinh.predict(X_kiem_tra)

    do_chinh_xac = accuracy_score(nhan_kiem_tra, nhan_du_doan)
    do_chinh_xac_duong = precision_score(
        nhan_kiem_tra, nhan_du_doan, zero_division=0
    )
    do_day_du = recall_score(
        nhan_kiem_tra, nhan_du_doan, zero_division=0
    )

    print(f"alpha = {he_so_alpha}")
    print(f" Do chinh xac (Accuracy)  = {do_chinh_xac:.4f}")
    print(f" Do chinh xac duong (Precision) = {do_chinh_xac_duong:.4f}")
    print(f" Do day du (Recall) = {do_day_du:.4f}")
    print()





alpha = 0.001
 Do chinh xac (Accuracy)  = 0.9467
 Do chinh xac duong (Precision) = 1.0000
 Do day du (Recall) = 0.8667

alpha = 0.1
 Do chinh xac (Accuracy)  = 0.9467
 Do chinh xac duong (Precision) = 1.0000
 Do day du (Recall) = 0.8667

alpha = 1.0
 Do chinh xac (Accuracy)  = 0.9467
 Do chinh xac duong (Precision) = 1.0000
 Do day du (Recall) = 0.8667

alpha = 10.0
 Do chinh xac (Accuracy)  = 0.9467
 Do chinh xac duong (Precision) = 1.0000
 Do day du (Recall) = 0.8667

alpha = 100.0
 Do chinh xac (Accuracy)  = 0.9467
 Do chinh xac duong (Precision) = 1.0000
 Do day du (Recall) = 0.8667



nhận xét: Khi alpha quá lớn, các xác suất xuất hiện của từ giữa các lớp trở nên gần nhau hơn, làm giảm khả năng phân biệt tin nhắn rác và tin nhắn thường, nên hiệu quả phân loại có thể giảm.

In [18]:
# câu 4:

import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Đọc kho 300 tin nhắn
df = pd.read_csv(
    "/Users/buixuankhanh/Documents/machine-learning/03_Navie_Bayes/data/tin_nhan.csv"
)

# Chia dữ liệu thành tập huấn luyện và tập kiểm tra
van_ban_huan_luyen, van_ban_kiem_tra, nhan_huan_luyen, nhan_kiem_tra = train_test_split(
    df["noi_dung"],
    df["la_rac"],
    test_size=0.25,
    random_state=17,
    stratify=df["la_rac"]
)

# Tạo từ điển chỉ từ tập huấn luyện
bo_tach_tu = CountVectorizer(token_pattern=r"(?u)\b\S+")
X_huan_luyen = bo_tach_tu.fit_transform(van_ban_huan_luyen)
X_kiem_tra = bo_tach_tu.transform(van_ban_kiem_tra)

tu_dien = bo_tach_tu.get_feature_names_out()
so_tu = len(tu_dien)

# Chuyển nhãn về kiểu số nguyên
nhan_huan_luyen = np.asarray(nhan_huan_luyen, dtype=int)
nhan_kiem_tra = np.asarray(nhan_kiem_tra, dtype=int)

# Tự đếm số lần mỗi từ xuất hiện trong từng lớp
dem = {0: Counter(), 1: Counter()}
tong_tu = {0: 0, 1: 0}
so_tin = {0: 0, 1: 0}

# Bộ tách từ giống bộ đếm từ ở trên
tach_tu = bo_tach_tu.build_analyzer()

for cau, nhan in zip(van_ban_huan_luyen, nhan_huan_luyen):
    cac_tu = tach_tu(cau)
    dem[nhan].update(cac_tu)
    tong_tu[nhan] += len(cac_tu)
    so_tin[nhan] += 1

print("So tu trong tu dien:", so_tu)
print("So tin huan luyen:", len(nhan_huan_luyen))
print("So tin kiem tra:", len(nhan_kiem_tra))
print()

# Tính log xác suất từ có làm trơn Laplace
def log_xac_suat(tu, nhan, alpha):
    return np.log(
        (dem[nhan][tu] + alpha)
        / (tong_tu[nhan] + alpha * so_tu)
    )

# Tính tổng loga cho mỗi lớp
def tinh_diem(cau, nhan, alpha):
    diem = np.log(so_tin[nhan] / len(nhan_huan_luyen))

    for tu, so_lan in Counter(tach_tu(cau)).items():
        if tu in bo_tach_tu.vocabulary_:
            diem += so_lan * log_xac_suat(tu, nhan, alpha)

    return diem

# Dự đoán bằng bộ lọc tự viết
alpha = 1.0
nhan_du_doan_tu_viet = []

for cau in van_ban_kiem_tra:
    diem_rac = tinh_diem(cau, 1, alpha)
    diem_thuong = tinh_diem(cau, 0, alpha)

    if diem_rac > diem_thuong:
        nhan_du_doan_tu_viet.append(1)
    else:
        nhan_du_doan_tu_viet.append(0)

# Huấn luyện MultinomialNB trên cùng tập dữ liệu
mo_hinh = MultinomialNB(alpha=alpha)
mo_hinh.fit(X_huan_luyen, nhan_huan_luyen)
nhan_du_doan_thu_vien = mo_hinh.predict(X_kiem_tra)

# Tính độ chính xác của hai mô hình
do_chinh_xac_tu_viet = accuracy_score(
    nhan_kiem_tra, nhan_du_doan_tu_viet
)
do_chinh_xac_thu_vien = accuracy_score(
    nhan_kiem_tra, nhan_du_doan_thu_vien
)

print("KET QUA SO SANH")
print(f"Accuracy tu viet       = {do_chinh_xac_tu_viet:.4f}")
print(f"Accuracy scikit-learn  = {do_chinh_xac_thu_vien:.4f}")
print(
    f"Do lech                = "
    f"{abs(do_chinh_xac_tu_viet - do_chinh_xac_thu_vien):.4f}"
)

So tu trong tu dien: 40
So tin huan luyen: 225
So tin kiem tra: 75

KET QUA SO SANH
Accuracy tu viet       = 0.9467
Accuracy scikit-learn  = 0.9467
Do lech                = 0.0000


nhận xét: Nếu hai kết quả lệch nhiều, cần kiểm tra những lỗi nào?
Cách đếm từ: Bộ lọc tự viết phải đếm số lần xuất hiện của từ, không chỉ đếm số tin nhắn chứa từ đó. Công thức làm trơn: Phải dùng đúng công thức Laplace. Xác suất tiên nghiệm: Phải dùng cùng cách tính xác suất của lớp. Nếu hai lớp không có số lượng tin nhắn bằng nhau, không nên mặc định xác suất mỗi lớp là 0,5.Từ điển: Chỉ tạo từ điển từ tập huấn luyện; tập kiểm tra phải được biến đổi bằng chính từ điển đó.
Chia dữ liệu: Hai mô hình phải dùng cùng random_state=17, cùng test_size, cùng stratify và cùng thứ tự dữ liệu.Xử lý từ lặp lại: Một từ xuất hiện nhiều lần trong tin nhắn phải đóng góp log xác suất tương ứng với số lần xuất hiện.

In [21]:
#câu 5:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Đọc dữ liệu sinh viên
df = pd.read_csv("/Users/buixuankhanh/Documents/machine-learning/03_Navie_Bayes/data/sinh_vien.csv")

# Chọn hai đặc trưng
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]

# Chia dữ liệu
X_huan_luyen, X_kiem_tra, y_huan_luyen, y_kiem_tra = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=17,
    stratify=y
)

# Huấn luyện MultinomialNB
mo_hinh = MultinomialNB()
mo_hinh.fit(X_huan_luyen, y_huan_luyen)

# Dự đoán và tính Accuracy
y_du_doan = mo_hinh.predict(X_kiem_tra)
do_chinh_xac = accuracy_score(y_kiem_tra, y_du_doan)

print(f"Accuracy MultinomialNB = {do_chinh_xac:.4f}")



Accuracy MultinomialNB = 0.8000


1. Chương trình có báo lỗi không?
Thông thường, chương trình vẫn chạy nếu dữ liệu hợp lệ và các giá trị đặc trưng không âm. MultinomialNB được thiết kế cho dữ liệu đếm hoặc tần suất, nhưng gio_on và diem_giua_ky là các đặc trưng số đo của sinh viên.
Mình chưa thể xác định chính xác Accuracy nếu chưa chạy trên file dữ liệu thực tế của bạn.
2. Vì sao dùng MultinomialNB ở đây sai về nguyên tắc?
MultinomialNB phù hợp với dữ liệu đếm, chẳng hạn số lần một từ xuất hiện trong tin nhắn.
gio_on và diem_giua_ky là các đặc trưng số liên tục hoặc số đo, không phải số lần đếm sự kiện.
GaussianNB phù hợp hơn vì mô hình giả định mỗi đặc trưng trong từng lớp tuân theo phân phối chuẩn.
Nhận xét: MultinomialNB vẫn có thể chạy và trả về Accuracy, nhưng điều đó không chứng minh mô hình phù hợp về mặt thống kê. Nên so sánh với GaussianNB trên cùng tập kiểm tra để đánh giá sự khác biệt.

In [22]:
#câu 6:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, recall_score

# 1. Đọc dữ liệu
df = pd.read_csv(
    "/Users/buixuankhanh/Documents/machine-learning/03_Navie_Bayes/data/tin_nhan.csv"
)

# 2. Tạo bộ dữ liệu mất cân bằng
tin_thuong = df[df["la_rac"] == 0]
tin_rac = df[df["la_rac"] == 1].head(20)

du_lieu_moi = pd.concat(
    [tin_thuong, tin_rac],
    ignore_index=True
)

print("Số tin thường:", len(tin_thuong))
print("Số tin rác:", len(tin_rac))
print("Tổng số tin:", len(du_lieu_moi))

# 3. Chia dữ liệu huấn luyện và kiểm tra
X_huan_luyen, X_kiem_tra, y_huan_luyen, y_kiem_tra = train_test_split(
    du_lieu_moi["noi_dung"],
    du_lieu_moi["la_rac"],
    test_size=0.25,
    random_state=17,
    stratify=du_lieu_moi["la_rac"]
)

# 4. Chuyển tin nhắn thành bảng đếm từ
bo_dem_tu = CountVectorizer(token_pattern=r"(?u)\b\S+")
X_huan_luyen = bo_dem_tu.fit_transform(X_huan_luyen)
X_kiem_tra = bo_dem_tu.transform(X_kiem_tra)

# 5. Huấn luyện mô hình Naive Bayes
mo_hinh = MultinomialNB()
mo_hinh.fit(X_huan_luyen, y_huan_luyen)

# 6. Dự đoán
y_du_doan = mo_hinh.predict(X_kiem_tra)

# 7. Đánh giá mô hình
do_chinh_xac = accuracy_score(y_kiem_tra, y_du_doan)
do_day_du_rac = recall_score(
    y_kiem_tra,
    y_du_doan,
    pos_label=1,
    zero_division=0
)

print(f"Accuracy = {do_chinh_xac:.4f}")
print(f"Recall lớp rác = {do_day_du_rac:.4f}")

Số tin thường: 180
Số tin rác: 20
Tổng số tin: 200
Accuracy = 0.9800
Recall lớp rác = 0.8000


Nhận xét bài tập
Accuracy là tỷ lệ dự đoán đúng trên tổng số tin nhắn kiểm tra.
Recall lớp rác là tỷ lệ tin nhắn rác thực tế mà mô hình phát hiện được.
Với bộ dữ liệu gồm 180 tin thường và 20 tin rác, dữ liệu bị mất cân bằng nghiêm trọng. Nếu mô hình dự đoán tất cả tin nhắn đều là tin thường, accuracy vẫn có thể đạt 90% trên toàn bộ dữ liệu, nhưng recall của lớp rác bằng 0%.
Kết luận: Accuracy không phải lúc nào cũng phản ánh đúng chất lượng bộ lọc. Trong bài này, cần xem thêm recall của lớp rác để biết mô hình bỏ sót bao nhiêu tin nhắn rác. Tuy nhiên, không nên chỉ dựa vào recall; cũng cần xem precision để biết có bao nhiêu tin thường bị nhận nhầm là tin rác.